In [2]:
import numpy as np
import pandas as pd
import sys
from sklearn.model_selection import StratifiedKFold
import os
import warnings
warnings.filterwarnings('ignore')
sys.path.append("_libs")
from select_features import FC_DimRed
from utils import load_cov_mats, load_atms, load_corr_mats

# A-priori analysis
Ranking nodes considering the whole dataset (code used to produce data for Fig. S1) 

In [26]:
while os.getcwd().split(os.sep)[-1] != "REDDI":
    os.chdir("../")
project_root = os.getcwd()  # should be .../REDDI

# Filtering procedure explained in paper 2.3

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

datasets = {
    "ATM": load_atms(zscore=1.6),
    "Covariance": load_cov_mats(),
    "Correlation": load_corr_mats(),
}

rankings = {}

for name, (X, y) in datasets.items():

    X = X[:, :78, :78]
    rankings[name] = []

    print(f"\n{name}")

    selector = FC_DimRed(eta_threshold=0.1, nb_nodes=78)
    selector.fit_transform(X, y, metric="eta-squared")

    rankings[name]=selector.node_select_+0.0

    print(f"Fold {name}: {selector.node_select_}")

rankings_good = {key:[] for key in rankings.keys()}
for key in rankings.keys():
    for i in range(78):
       rankings_good[key].append(np.where(rankings[key] == i)[0][0])
    rankings_good[key] = np.array(rankings_good[key])+1.0
pd_rankings = pd.DataFrame.from_dict(rankings_good, orient="index")

Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)
Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)
Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)

ATM
Fold ATM: [36 56 57 63 38 64 49 15 14 54 25 52 65 11  6 17 59  1 42 47 51 58 75 60
 18  8 53 20 61 28 46 23 50 37 76 77 27 10 22 24 29 12 55 19 21 26 66 62
 45 32 74  7 43 31 40 35 13 71 72 73  3 67 16  0  9 41  5 48 30 68 39  4
 69 70 44 33  2 34]

Covariance
Fold Covariance: [77 11 73 51 33 26 27 62 50 14 72 67 36 34 18 37 12 44  8  5 71 15 24 31
 39 38 23  9 63 45 76 70 65 40 13  7 17 75  1 25 56 20 59 43 32 21 66 52
 22 60 47 46 74 54 48 16  6 69 53 55 64 41 30 61 57 58 19 49 42  0 29 35
  4  3  2 10 28 68]

Correlation
Fold Correlation: [77 67 56 20 38 17 51 50 62 12  6 54 75 11 73 18 57 45  7 65 36 33  5 63
 32 61 39 42 52 13 69 2

In [27]:
rankings_good

{'ATM': array([64., 18., 77., 61., 72., 67., 15., 52., 26., 65., 38., 14., 42.,
        57.,  9.,  8., 63., 16., 25., 44., 28., 45., 39., 32., 40., 11.,
        46., 37., 30., 41., 69., 54., 50., 76., 78., 56.,  1., 34.,  5.,
        71., 55., 66., 19., 53., 75., 49., 31., 20., 68.,  7., 33., 21.,
        12., 27., 10., 43.,  2.,  3., 22., 17., 24., 29., 48.,  4.,  6.,
        13., 47., 62., 70., 73., 74., 58., 59., 60., 51., 23., 35., 36.]),
 'Covariance': array([70., 39., 75., 74., 73., 20., 57., 36., 19., 28., 76.,  2., 17.,
        35., 10., 22., 56., 37., 15., 67., 42., 46., 49., 27., 23., 40.,
         6.,  7., 77., 71., 63., 24., 45.,  5., 14., 72., 13., 16., 26.,
        25., 34., 62., 69., 44., 18., 30., 52., 51., 55., 68.,  9.,  4.,
        48., 59., 54., 60., 41., 65., 66., 43., 50., 64.,  8., 29., 61.,
        33., 47., 12., 78., 58., 32., 21., 11.,  3., 53., 38., 31.,  1.]),
 'Correlation': array([78., 46., 77., 67., 72., 23., 11., 19., 33., 48., 57., 14., 10.,
        30.

In [30]:
import scipy
scipy.io.savemat("ranks_whole_dataset.mat", rankings_good)

# Per-fold selection
Code used to produce data for Fig. 3 and Fig. S2 

## 

In [3]:
while os.getcwd().split(os.sep)[-1] != "REDDI":
    os.chdir("../")
project_root = os.getcwd()  # should be .../REDDI

# For each fold i perform the selection of the features (Filtering procedure explained in paper 2.3)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

datasets = {
    "ATM": load_atms(zscore=1.6),
    "Covariance": load_cov_mats(),
    "Correlation": load_corr_mats(),
}

rankings = {}

for name, (X, y) in datasets.items():

    X = X[:, :78, :78]
    rankings[name] = []

    print(f"\n{name}")

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):

        selector = FC_DimRed(eta_threshold=0.1, nb_nodes=78)
        selector.fit_transform(X[train_idx], y[train_idx], metric="eta-squared")

        rankings[name].append(selector.node_select_)

        print(f"Fold {fold}: {selector.node_select_}")

pd_rankings = pd.DataFrame.from_dict(rankings, orient="index")

Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)
Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)
Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)

ATM
Fold 1: [36 54 51 64 57 56 52 25 38 47 14 11 15 46 17 77  1 66 65 18 53 58 63 67
 28 55 60  6 12 61 24 62 49  3  8 73 29 19 20 23  7 13 59 27 37 42 10 45
 40 26 74 75 50 35 68 16  5 21  9  4 30 72 41 76 31 22 71 70 69 33 39 43
 48  2 32 34 44  0]
Fold 2: [36 64 49 15 38 25 63  6 58 56 17 54 59 65 75 57 52 18 51 76 14 20 50 74
 37 11 61 32 77 12  1 19 10 21 60 35 43 42 13 16 45 28 29  8 62 66 22  9
  7 55 53 41 40 26 27 47 73 72 71 67  5  0 24 46  3 31 23 70  2  4 48 68
 39 44 34 69 30 33]
Fold 3: [49 74 42 64 63 50  1 75 10 37 65 59 36 25  0 38 14 57 51 15 35 18 20 54
 17 56 52 21 12 24  6 28 76 47  3 53 27 11 66 16 60 61 67 58 45 77 26

In [4]:
mean_ranks = {}
std_ranks = {}

for modality, folds in rankings.items():

    # matrice: righe = fold, colonne = nodi
    rank_matrix = np.zeros((len(folds), 78))

    for i, ranking in enumerate(folds):
        for rank, node in enumerate(ranking, start=1):
            rank_matrix[i, node] = rank

    mean_ranks[modality] = pd.Series(
        rank_matrix.mean(axis=0),
        index=np.arange(78),
        name=modality
    )
    std_ranks[modality] = pd.Series(
        rank_matrix.std(axis=0),
        index=np.arange(78),
        name=modality
    )

pd_mean_ranks = pd.DataFrame(mean_ranks)
pd_std_ranks = pd.DataFrame(std_ranks)

In [5]:
pd_std_ranks["ATM"][36]

4.874423042781576

In [ ]:
# check for how many folds each region was selected in the top 50 for each modality

ATM_rankings = []
COV_rankings = []
CORR_rankings = []
for i in range(5):
    ATM_rankings.append(pd_rankings.loc["ATM", i])
    COV_rankings.append(pd_rankings.loc["Covariance", i])
    CORR_rankings.append(pd_rankings.loc["Correlation", i])
pd_rankings[0]["ATM"]

how_many_times_selected_ATM = np.zeros((78))
for rank_list in ATM_rankings:
    for region,rank in enumerate(rank_list):
        if rank < 50:
            how_many_times_selected_ATM[region] += 1

how_many_times_selected_COV = np.zeros((78))
for rank_list in COV_rankings:
    for region,rank in enumerate(rank_list):
        if rank < 50:
            how_many_times_selected_COV[region] += 1

how_many_times_selected_CORR = np.zeros((78))
for rank_list in CORR_rankings:
    for region,rank in enumerate(rank_list):
        if rank < 50:
            how_many_times_selected_CORR[region] += 1
how_many_dict = {"ATM": how_many_times_selected_ATM, "Covariance": how_many_times_selected_COV, "Correlation": how_many_times_selected_CORR}

In [29]:
how_many_dict["Correlation"]

array([0., 0., 2., 3., 1., 1., 3., 4., 3., 3., 3., 4., 3., 4., 3., 3., 4.,
       4., 3., 0., 1., 3., 3., 3., 3., 3., 5., 3., 2., 4., 2., 3., 2., 2.,
       3., 4., 4., 4., 3., 4., 3., 3., 2., 3., 3., 4., 3., 2., 3., 4., 4.,
       4., 4., 4., 4., 5., 3., 4., 3., 5., 4., 3., 3., 4., 4., 4., 5., 2.,
       5., 4., 5., 4., 4., 2., 4., 5., 3., 3.])

In [ ]:
# List of the 78 ROIs in the AAL atlas, in the same order as the matrices

ROI_AAL_list = np.array([ 'Rectus_L','Olfactory_L','Frontal_Sup_Orb_L','Frontal_Med_Orb_L','Frontal_Mid_Orb_L',
                'Frontal_Inf_Orb_L','Frontal_Sup_L','Frontal_Mid_L','Frontal_Inf_Oper_L','Frontal_Inf_Tri_L',
                'Frontal_Sup_Medial_L','Supp_Motor_Area_L','Paracentral_Lobule_L','Precentral_L','Rolandic_Oper_L',
                'Postcentral_L','Parietal_Sup_L','Parietal_Inf_L','SupraMarginal_L','Angular_L','Precuneus_L',
                'Occipital_Sup_L','Occipital_Mid_L','Occipital_Inf_L','Calcarine_L','Cuneus_L','Lingual_L',
                'Fusiform_L','Heschl_L','Temporal_Sup_L','Temporal_Mid_L','Temporal_Inf_L','Temporal_Pole_Sup_L',
                'Temporal_Pole_Mid_L','ParaHippocampal_L','Cingulum_Ant_L','Cingulum_Mid_L','Cingulum_Post_L',
                'Insula_L','Rectus_R','Olfactory_R','Frontal_Sup_Orb_R','Frontal_Med_Orb_R','Frontal_Mid_Orb_R',
                'Frontal_Inf_Orb_R','Frontal_Sup_R','Frontal_Mid_R','Frontal_Inf_Oper_R','Frontal_Inf_Tri_R',
                'Frontal_Sup_Medial_R','Supp_Motor_Area_R','Paracentral_Lobule_R','Precentral_R','Rolandic_Oper_R',
                'Postcentral_R','Parietal_Sup_R','Parietal_Inf_R', 'SupraMarginal_R','Angular_R','Precuneus_R',
                'Occipital_Sup_R','Occipital_Mid_R','Occipital_Inf_R','Calcarine_R','Cuneus_R','Lingual_R',
                'Fusiform_R','Heschl_R','Temporal_Sup_R','Temporal_Mid_R','Temporal_Inf_R','Temporal_Pole_Sup_R',
                'Temporal_Pole_Mid_R','ParaHippocampal_R','Cingulum_Ant_R','Cingulum_Mid_R','Cingulum_Post_R',
                'Insula_R','Hippocampus_L','Hippocampus_R','Amygdala_L','Amygdala_R','Caudate_L','Caudate_R',
                'Putamen_L','Putamen_R','Pallidum_L','Pallidum_R','Thalamus_L','Thalamus_R','Cerebelum_Crus1_L',
                'Cerebelum_Crus1_R','Cerebelum_Crus2_L','Cerebelum_Crus2_R','Cerebelum_3_L','Cerebelum_3_R',
                'Cerebelum_4_5_L','Cerebelum_4_5_R','Cerebelum_6_L','Cerebelum_6_R','Cerebelum_7b_L','Cerebelum_7b_R',
                'Cerebelum_8_L','Cerebelum_8_R','Cerebelum_9_L','Cerebelum_9_R','Cerebelum_10_L','Cerebelum_10_R',
                'Vermis_1_2','Vermis_3','Vermis_4_5','Vermis_6','Vermis_7','Vermis_8','Vermis_9','Vermis_10'])